# Préprocessing et Feature Engineering — NBA Shot Analysis

## Objectif

Cette étape a pour objectif de préparer le jeu de données issu de l'EDA pour les analyses approfondies et la modélisation de la probabilité de réussite des tirs.

Suite aux constats réalisés lors de l'EDA, nous allons notamment :

- restreindre l'étude aux saisons à partir de 2012-13 ;
- supprimer les doublons exacts ;
- traiter les valeurs manquantes ;
- convertir les variables dans des formats adaptés ;
- supprimer certaines variables redondantes ;
- créer de nouvelles variables utiles à l'analyse et à la modélisation ;
- effectuer plusieurs contrôles de qualité ;
- exporter le dataset prétraité.

> **Périmètre temporel :** saisons 2012-13 à 2024-25.
>
> Dans le dataset, la saison 2012-13 correspond à `SEASON_1 = 2013`.


## 1. Import des bibliothèques

Nous utilisons DuckDB afin de travailler efficacement avec les fichiers CSV sans devoir charger simultanément l'ensemble des données brutes en mémoire.

In [48]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 2. Chargement des données

Les données brutes sont stockées dans le dossier `data/`.

Le dossier local `data/` est monté dans le conteneur Docker sous `/app/data`. Nous utilisons donc ce chemin absolu afin que le notebook fonctionne correctement dans l'environnement Docker du projet.

In [49]:
DATA = "/app/data/*.csv"

_SRC = f'read_csv_auto("{DATA}", union_by_name=true)'

con = duckdb.connect()

def q(sql):
    return con.execute(sql.replace("{DATA}", _SRC)).df()

In [50]:
import os
from pathlib import Path

print("Répertoire courant :", os.getcwd())

print("\nContenu de /app :")
print(os.listdir("/app"))

print("\nContenu de /app/data :")
print(os.listdir("/app/data"))

Répertoire courant : /app

Contenu de /app :
['eda.ipynb', 'expectations.md', '.DS_Store', 'datasets_ref.md', 'Dockerfile', 'output', 'README.md', '.dockerignore', '.gitignore', 'docker-compose.yml', '.ipynb_checkpoints', '.git', 'data', 'src']

Contenu de /app/data :
['NBA_2009_Shots.csv', 'NBA_2025_Shots.csv', 'NBA_2004_Shots.csv', 'NBA_2010_Shots.csv', '.DS_Store', 'NBA_2016_Shots.csv', 'NBA_2023_Shots.csv', 'NBA_2008_Shots.csv', 'NBA_2024_Shots.csv', 'NBA_2011_Shots.csv', 'NBA_2005_Shots.csv', 'NBA_2017_Shots.csv', 'NBA_2022_Shots.csv', 'NBA_2006_Shots.csv', 'NBA_2012_Shots.csv', 'NBA_2019_Shots.csv', 'NBA_2021_Shots.csv', 'NBA_2014_Shots.csv', 'NBA_2013_Shots.csv', 'NBA_2007_Shots.csv', 'NBA_2018_Shots.csv', 'NBA_2020_Shots.csv', 'NBA_2015_Shots.csv']


## 3. Vérification du périmètre des données

Avant de commencer le preprocessing, nous vérifions les saisons disponibles dans les fichiers bruts.

In [51]:
seasons = q("""
    SELECT
        SEASON_1,
        SEASON_2,
        COUNT(*) AS NB_SHOTS
    FROM {DATA}
    GROUP BY SEASON_1, SEASON_2
    ORDER BY SEASON_1
""")

seasons

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,SEASON_1,SEASON_2,NB_SHOTS
0,2004,2003-04,189803
1,2005,2004-05,197626
2,2006,2005-06,194314
3,2007,2006-07,196072
4,2008,2007-08,200501
5,2009,2008-09,199030
6,2010,2009-10,200966
7,2011,2010-11,199761
8,2012,2011-12,161205
9,2013,2012-13,201579


### Choix du périmètre temporel

L'EDA a mis en évidence une rupture dans les données historiques concernant notamment les coordonnées et la distance des tirs.

Afin de limiter l'impact des changements de règles et de collecte des données, nous retenons les saisons **à partir de 2012-13**.

Dans le dataset :

- `SEASON_1 = 2013` correspond à la saison `2012-13` ;
- nous conservons donc les observations pour lesquelles `SEASON_1 >= 2013`.

In [52]:
df = q("""
    SELECT *
    FROM {DATA}
    WHERE SEASON_1 >= 2013
""")

print(f"Nombre de tirs après filtrage : {len(df):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Nombre de tirs après filtrage : 2,711,511


In [53]:
df[["SEASON_1", "SEASON_2"]].drop_duplicates().sort_values("SEASON_1")

,SEASON_1,SEASON_2
0,2013,2012-13
201579,2014,2013-14
405705,2015,2014-15
611255,2016,2015-16
819148,2017,2016-17
1029077,2018,2017-18
1240784,2019,2018-19
1460242,2020,2019-20
1648358,2021,2020-21
1839341,2022,2021-22


## 4. Taille du dataset

Nous vérifions le nombre d'observations et de variables conservées après le filtrage temporel.

In [54]:
print("Nombre de lignes :", f"{df.shape[0]:,}")
print("Nombre de colonnes :", df.shape[1])

Nombre de lignes : 2,711,511
Nombre de colonnes : 26


## 5. Suppression des doublons

L'EDA a identifié des doublons exacts dans le jeu de données.

Les doublons exacts sont supprimés afin qu'une même observation ne soit pas comptabilisée plusieurs fois dans les analyses et la modélisation.

In [55]:
print("Doublons avant suppression :", df.duplicated().sum())

df = df.drop_duplicates().copy()

print("Doublons après suppression :", df.duplicated().sum())
print("Nombre de lignes :", f"{len(df):,}")

Doublons avant suppression : 99
Doublons après suppression : 0
Nombre de lignes : 2,711,412


## 6. Analyse des valeurs manquantes

Nous commençons par identifier les variables contenant des valeurs manquantes.

In [56]:
missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("NB_NA")
)

missing["PCT_NA"] = (missing["NB_NA"] / len(df) * 100).round(2)

missing[missing["NB_NA"] > 0]

,NB_NA,PCT_NA
POSITION_GROUP,219517,8.1
POSITION,219517,8.1


### Traitement des positions

L'EDA montre la présence de valeurs manquantes dans `POSITION` et `POSITION_GROUP`, notamment sur les données les plus récentes.

Dans un premier temps, nous ne supprimons pas ces observations afin d'éviter une perte importante de tirs.

Les valeurs manquantes sont remplacées par la catégorie `"Unknown"`.

Une éventuelle amélioration pourra consister à enrichir ces informations avec `nba_api` à partir de `PLAYER_ID`.

In [57]:
df["POSITION"] = df["POSITION"].replace("NA", np.nan)
df["POSITION_GROUP"] = df["POSITION_GROUP"].replace("NA", np.nan)

df["POSITION"] = df["POSITION"].fillna("Unknown")
df["POSITION_GROUP"] = df["POSITION_GROUP"].fillna("Unknown")

## 7. Conversion des types

`GAME_DATE` est initialement stockée sous forme de texte.

Nous la convertissons en objet datetime afin de pouvoir effectuer ultérieurement des analyses temporelles.

In [58]:
df["GAME_DATE"] = pd.to_datetime(
    df["GAME_DATE"],
    format="%m-%d-%Y",
    errors="coerce"
)

df["GAME_DATE"].dtype

dtype('<M8[ns]')

In [59]:
print("Dates invalides :", df["GAME_DATE"].isna().sum())

Dates invalides : 0


### Variables numériques

Nous nous assurons que les principales variables quantitatives sont bien interprétées comme numériques.

In [60]:
numeric_cols = [
    "LOC_X",
    "LOC_Y",
    "SHOT_DISTANCE",
    "QUARTER",
    "MINS_LEFT",
    "SECS_LEFT",
    "SEASON_1",
    "PLAYER_ID",
    "TEAM_ID",
    "GAME_ID"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

## 8. Variable cible : SHOT_MADE

La variable `SHOT_MADE` indique si le tir a été réussi :

- `0` : tir manqué ;
- `1` : tir réussi.

Cette variable constituera la variable cible pour la modélisation de la probabilité de réussite d'un tir.

In [61]:
df["SHOT_MADE"] = pd.to_numeric(
    df["SHOT_MADE"],
    errors="coerce"
).astype("Int64")

df["SHOT_MADE"].value_counts(dropna=False)

SHOT_MADE
0       1461523
1       1249889
<NA>          0
Name: count, dtype: Int64

## 9. Suppression des variables redondantes

L'EDA a permis d'identifier plusieurs variables contenant une information redondante.

### `EVENT_TYPE`

`EVENT_TYPE` est parfaitement redondante avec `SHOT_MADE`. Nous conservons donc `SHOT_MADE`, qui est directement exploitable comme variable cible.

### `SEASON_2`

`SEASON_2` contient la même information que `SEASON_1`, mais sous une forme textuelle (`2012-13`, `2013-14`, etc.).

Nous conservons `SEASON_1` pour les traitements numériques.

### `ZONE_ABB`

`ZONE_ABB` est une représentation abrégée de `ZONE_NAME`. Nous conservons `ZONE_NAME`.

In [62]:
cols_to_drop = [
    "EVENT_TYPE",
    "SEASON_2",
    "ZONE_ABB"
]

cols_to_drop = [col for col in cols_to_drop if col in df.columns]

df = df.drop(columns=cols_to_drop)

print("Variables supprimées :", cols_to_drop)
print("Nombre de colonnes restantes :", df.shape[1])

Variables supprimées : ['EVENT_TYPE', 'SEASON_2', 'ZONE_ABB']
Nombre de colonnes restantes : 23


## 10. Feature Engineering

Nous créons plusieurs variables supplémentaires susceptibles d'être utiles pour l'analyse et la modélisation.

### Temps restant

Les variables `MINS_LEFT` et `SECS_LEFT` représentent le temps restant dans la période.

Nous créons une variable unique `TIME_REMAINING_SEC`, exprimée en secondes.

In [63]:
df["TIME_REMAINING_SEC"] = (
    df["MINS_LEFT"] * 60 + df["SECS_LEFT"]
)

df["TIME_REMAINING_SEC"].describe()

count    2.711412e+06
mean     3.499202e+02
std      2.081932e+02
min      0.000000e+00
25%      1.690000e+02
50%      3.510000e+02
75%      5.310000e+02
max      7.200000e+02
Name: TIME_REMAINING_SEC, dtype: float64

In [64]:
df[
    [
        "QUARTER",
        "MINS_LEFT",
        "SECS_LEFT",
        "TIME_REMAINING_SEC"
    ]
].head(10)

,QUARTER,MINS_LEFT,SECS_LEFT,TIME_REMAINING_SEC
0,5,0,5,5
1,5,0,20,20
2,5,0,47,47
3,5,0,56,56
4,5,1,11,71
5,5,1,19,79
6,5,1,34,94
7,5,1,57,117
8,5,2,26,146
9,4,0,15,15


## 11. Contrôles de qualité

Nous effectuons plusieurs vérifications avant d'exporter le dataset final :

- absence de doublons exacts ;
- valeurs de `SHOT_MADE` cohérentes ;
- absence de valeurs manquantes critiques ;
- cohérence des saisons ;
- vérification des principales variables numériques.

In [68]:
print("=== CONTRÔLES ===")

print("\nDoublons :", df.duplicated().sum())

print("\nSHOT_MADE :")
print(df["SHOT_MADE"].value_counts(dropna=False))

print("\nSaisons :")
print(
    df["SEASON_1"]
    .value_counts()
    .sort_index()
)

print("\nValeurs manquantes :")
print(
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .head(20)
)

=== CONTRÔLES ===

Doublons : 0

SHOT_MADE :
SHOT_MADE
0       1461523
1       1249889
<NA>          0
Name: count, dtype: Int64

Saisons :
SEASON_1
2013    201579
2014    204125
2015    205550
2016    207893
2017    209929
2018    211693
2019    219443
2020    188100
2021    190971
2022    216718
2023    217207
2024    218687
2025    219517
Name: count, dtype: int64

Valeurs manquantes :
SEASON_1          0
TEAM_ID           0
SECS_LEFT         0
MINS_LEFT         0
QUARTER           0
SHOT_DISTANCE     0
LOC_Y             0
LOC_X             0
ZONE_RANGE        0
ZONE_NAME         0
BASIC_ZONE        0
SHOT_TYPE         0
ACTION_TYPE       0
SHOT_MADE         0
AWAY_TEAM         0
HOME_TEAM         0
GAME_ID           0
GAME_DATE         0
POSITION          0
POSITION_GROUP    0
dtype: int64


## 12. Dataset final

Nous affichons quelques informations générales sur le dataset après preprocessing.

In [66]:
print("Dimensions finales :", df.shape)

display(df.head(20))

display(df.dtypes)

Dimensions finales : (2711412, 24)


,SEASON_1,TEAM_ID,TEAM_NAME,PLAYER_ID,PLAYER_NAME,POSITION_GROUP,POSITION,GAME_DATE,GAME_ID,HOME_TEAM,AWAY_TEAM,SHOT_MADE,ACTION_TYPE,SHOT_TYPE,BASIC_ZONE,ZONE_NAME,ZONE_RANGE,LOC_X,LOC_Y,SHOT_DISTANCE,QUARTER,MINS_LEFT,SECS_LEFT,TIME_REMAINING_SEC
0,2013,1610612745,Houston Rockets,2568,Carlos Delfino,F,SF,2013-04-17,21201228,LAL,HOU,0,Jump Shot,2PT Field Goal,Mid-Range,Right Side,16-24 ft.,-21.7,6.65,21,5,0,5,5
1,2013,1610612745,Houston Rockets,201935,James Harden,G,SG,2013-04-17,21201228,LAL,HOU,0,Jump Shot,2PT Field Goal,In The Paint (Non-RA),Center,Less Than 8 ft.,-3.6,7.75,4,5,0,20,20
2,2013,1610612745,Houston Rockets,201935,James Harden,G,SG,2013-04-17,21201228,LAL,HOU,0,Jump Shot,3PT Field Goal,Above the Break 3,Left Side Center,24+ ft.,17.1,23.65,25,5,0,47,47
3,2013,1610612747,Los Angeles Lakers,201975,Jodie Meeks,G,SG,2013-04-17,21201228,LAL,HOU,1,Driving Dunk Shot,2PT Field Goal,Restricted Area,Center,Less Than 8 ft.,-1.3,5.85,1,5,0,56,56
4,2013,1610612745,Houston Rockets,201935,James Harden,G,SG,2013-04-17,21201228,LAL,HOU,1,Driving Finger Roll Layup Shot,2PT Field Goal,Restricted Area,Center,Less Than 8 ft.,0.5,7.45,2,5,1,11,71
5,2013,1610612747,Los Angeles Lakers,2730,Dwight Howard,C,C,2013-04-17,21201228,LAL,HOU,1,Hook Shot,2PT Field Goal,In The Paint (Non-RA),Center,Less Than 8 ft.,2.1,9.35,4,5,1,19,79
6,2013,1610612745,Houston Rockets,202718,Chandler Parsons,F,SF,2013-04-17,21201228,LAL,HOU,0,Jump Shot,2PT Field Goal,Mid-Range,Left Side,8-16 ft.,8.4,7.85,8,5,1,34,94
7,2013,1610612747,Los Angeles Lakers,201975,Jodie Meeks,G,SG,2013-04-17,21201228,LAL,HOU,0,Jump Shot,3PT Field Goal,Left Corner 3,Left Side,24+ ft.,23.7,4.45,23,5,1,57,117
8,2013,1610612747,Los Angeles Lakers,2200,Pau Gasol,F,PF,2013-04-17,21201228,LAL,HOU,1,Turnaround Jump Shot,2PT Field Goal,In The Paint (Non-RA),Center,Less Than 8 ft.,0.5,12.75,7,5,2,26,146
9,2013,1610612756,Phoenix Suns,201632,Hamed Haddadi,C,C,2013-04-17,21201227,DEN,PHX,1,Reverse Dunk Shot,2PT Field Goal,Restricted Area,Center,Less Than 8 ft.,-0.9,6.75,1,4,0,15,15


SEASON_1                       int64
TEAM_ID                        int64
TEAM_NAME                     object
PLAYER_ID                      int64
PLAYER_NAME                   object
POSITION_GROUP                object
POSITION                      object
GAME_DATE             datetime64[ns]
GAME_ID                        int64
HOME_TEAM                     object
AWAY_TEAM                     object
SHOT_MADE                      Int64
ACTION_TYPE                   object
SHOT_TYPE                     object
BASIC_ZONE                    object
ZONE_NAME                     object
ZONE_RANGE                    object
LOC_X                        float64
LOC_Y                        float64
SHOT_DISTANCE                  int64
QUARTER                        int64
MINS_LEFT                      int64
SECS_LEFT                      int64
TIME_REMAINING_SEC             int64
dtype: object

## 13. Export du dataset préprocessed

Le dataset nettoyé est enregistré au format Parquet.

Le format Parquet est privilégié pour la suite du projet car il est plus compact et permet des lectures efficaces, notamment avec DuckDB.

In [69]:
import os

os.makedirs("/app/data/processed", exist_ok=True)

output_path = "/app/data/processed/shots_preprocessed_2012_2025.parquet"

df.to_parquet(
    output_path,
    index=False
)

print(f"Dataset sauvegardé : {output_path}")

Dataset sauvegardé : /app/data/processed/shots_preprocessed_2012_2025.parquet


In [70]:
df_check = pd.read_parquet(output_path)

print("Dimensions du fichier exporté :", df_check.shape)

df_check.head()

Dimensions du fichier exporté : (2711412, 24)


,SEASON_1,TEAM_ID,TEAM_NAME,PLAYER_ID,PLAYER_NAME,POSITION_GROUP,POSITION,GAME_DATE,GAME_ID,HOME_TEAM,AWAY_TEAM,SHOT_MADE,ACTION_TYPE,SHOT_TYPE,BASIC_ZONE,ZONE_NAME,ZONE_RANGE,LOC_X,LOC_Y,SHOT_DISTANCE,QUARTER,MINS_LEFT,SECS_LEFT,TIME_REMAINING_SEC
0,2013,1610612745,Houston Rockets,2568,Carlos Delfino,F,SF,2013-04-17,21201228,LAL,HOU,0,Jump Shot,2PT Field Goal,Mid-Range,Right Side,16-24 ft.,-21.7,6.65,21,5,0,5,5
1,2013,1610612745,Houston Rockets,201935,James Harden,G,SG,2013-04-17,21201228,LAL,HOU,0,Jump Shot,2PT Field Goal,In The Paint (Non-RA),Center,Less Than 8 ft.,-3.6,7.75,4,5,0,20,20
2,2013,1610612745,Houston Rockets,201935,James Harden,G,SG,2013-04-17,21201228,LAL,HOU,0,Jump Shot,3PT Field Goal,Above the Break 3,Left Side Center,24+ ft.,17.1,23.65,25,5,0,47,47
3,2013,1610612747,Los Angeles Lakers,201975,Jodie Meeks,G,SG,2013-04-17,21201228,LAL,HOU,1,Driving Dunk Shot,2PT Field Goal,Restricted Area,Center,Less Than 8 ft.,-1.3,5.85,1,5,0,56,56
4,2013,1610612745,Houston Rockets,201935,James Harden,G,SG,2013-04-17,21201228,LAL,HOU,1,Driving Finger Roll Layup Shot,2PT Field Goal,Restricted Area,Center,Less Than 8 ft.,0.5,7.45,2,5,1,11,71


# Conclusion

Le preprocessing a permis de construire un dataset propre et cohérent pour les prochaines étapes du projet.

### Étapes réalisées

- Restriction aux saisons **2012-13 à 2024-25** ;
- suppression des doublons exacts ;
- traitement des valeurs manquantes ;
- conversion des variables dans des formats adaptés ;
- suppression des variables redondantes ;
- conservation des identifiants `PLAYER_ID`, `GAME_ID` et `TEAM_ID` ;
- création de `TIME_REMAINING_SEC` à partir des variables temporelles existantes ;
- export du dataset prétraité au format Parquet.

Des features supplémentaires pourront être testées lors de la phase de modélisation si l'analyse exploratoire ou les performances des modèles le justifient.

L'enrichissement avec `nba_api` pourra également être réalisé dans une étape distincte, notamment à partir de `PLAYER_ID`.